# 04 · Parse PDFs into typed elements

> **Run order.** This notebook is step 4 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


Turns each PDF into typed elements — text, headings, tables, figures — each
carrying `document_id / page / element_id / bbox`.

**Ordering rule.** Tables are located *first* and their page regions claimed,
then text blocks overlapping a claimed region are skipped. Without that, every
number in every table appears twice — once as a structured cell, once as mangled
loose text — and the retriever ranks the mangled copy above the structured one.

Parser choice was decided by measurement, not reputation — see notebook 05 and
[ADR-004](../docs/adr/0004-pdf-parser.md).

In [ ]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from pathlib import Path
from sqlalchemy import select
from analyst.db import session_scope
from analyst.models import Document

with session_scope() as s:
    docs = [(d.document_id, d.ticker, d.fiscal_year, Path(d.local_path))
            for d in s.execute(select(Document).order_by(Document.ticker)).scalars().all()]
pd.DataFrame([{"document_id": d[0], "ticker": d[1], "fy": d[2]} for d in docs])

## Look at one page before trusting the whole corpus

Any parser will produce *something*. The question is whether the something is usable.

In [ ]:
from analyst.parsing import extract_elements
from analyst.config import get_settings

settings = get_settings()
doc_id, ticker, fy, path = next(d for d in docs if d[1] == "SUNPHARMA" and d[2] == 2025)

sample = list(extract_elements(path, doc_id, settings.data_dir / "figures" / doc_id,
                               max_pages=8, with_figures=False))
pd.DataFrame([
    {"element_id": e.element_id.split(":", 1)[1], "page": e.page, "type": e.type.value,
     "chars": len(e.text or ""), "preview": (e.text or "")[:70].replace("\n", " ")}
    for e in sample
]).head(12)

## A table keeps its structure, not just its text

In [ ]:
table = next(e for e in sample if e.type.value == "table" and e.table_json)
print("header:", table.table_json["header"])
print("rows  :", table.table_json["n_rows"])
print("page  :", table.page, " bbox:", table.bbox)
pd.DataFrame(table.table_json["rows"], columns=table.table_json["header"]).head()

## Parse the whole corpus

Re-runnable: element IDs are deterministic, so this overwrites in place rather than duplicating, and any index built on top stays valid.

In [ ]:
from sqlalchemy import update
from sqlalchemy.dialects.postgresql import insert
from analyst.models import ElementRow
from analyst.parsing import page_count
from analyst.provenance import ElementType

CHUNK = 1000
figure_dir = settings.data_dir / "figures"

def to_row(el):
    return {"element_id": el.element_id, "document_id": el.document_id, "page": el.page,
            "type": el.type.value, "seq": el.order, "text": el.text,
            "table_json": el.table_json, "image_path": el.image_path,
            "bbox": list(el.bbox) if el.bbox else None}

def flush(session, rows):
    stmt = insert(ElementRow).values(rows)
    session.execute(stmt.on_conflict_do_update(
        index_elements=[ElementRow.element_id],
        set_={"text": stmt.excluded.text, "table_json": stmt.excluded.table_json,
              "image_path": stmt.excluded.image_path, "bbox": stmt.excluded.bbox,
              "type": stmt.excluded.type}))

report = []
for document_id, ticker, fy, path in docs:
    if not path.exists():
        continue
    counts = {t.value: 0 for t in ElementType}
    batch, total = [], 0
    with session_scope() as s:
        for el in extract_elements(path, document_id, figure_dir / document_id):
            counts[el.type.value] += 1
            batch.append(to_row(el))
            total += 1
            if len(batch) >= CHUNK:
                flush(s, batch)
                batch = []
        if batch:
            flush(s, batch)
        s.execute(update(Document).where(Document.document_id == document_id)
                  .values(n_pages=page_count(path)))
    report.append({"ticker": ticker, "fy": fy, "pages": page_count(path),
                   "elements": total, **counts})

df = pd.DataFrame(report).drop(columns=["caption"], errors="ignore")
print(f"TOTAL ELEMENTS: {df['elements'].sum():,}")
df